# Statistical Tests Notebook - A2-V4

This notebook satisfies the mandatory statistical section from README:
1. Wilcoxon signed-rank test comparing two segmentation models
2. Bootstrap (B=1000) confidence interval for best model mIoU

Models used in this comparison:
- Model A: CustomSegmentationNet (ResNet50 backbone + custom decoder), checkpoint `best_segmentation_a.pth`
- Model B: CustomSegmentationNet (ResNet50 backbone + custom decoder), checkpoint `best_segmentation_b.pth`

Note: since true test masks are unavailable, this uses a held-out split from train data.
The architecture below matches the custom segmentation model used in training_notebook.ipynb.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import List, Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.transforms import functional as TF

ROOT = Path('.').resolve()
KAGGLE = Path('/kaggle').exists()

def _find_kaggle_dataset_root() -> Path:
    candidates = [
        Path('/kaggle/input/datasets/sinfilroamigo/mlds-a2/Dataset'),
        Path('/kaggle/input/mlds-a2/Dataset'),
    ]
    for c in candidates:
        if (c / 'train').exists() and (c / 'test').exists():
            return c
    for p in Path('/kaggle/input').rglob('Dataset'):
        if (p / 'train').exists() and (p / 'test').exists():
            return p
    raise FileNotFoundError('Could not locate Dataset/ under /kaggle/input.')

if KAGGLE:
    DATA_ROOT = _find_kaggle_dataset_root()
    SUBMISSION_ROOT = Path('/kaggle/working') / 'Roll_no_name_Assignment_2'
    SEG_WEIGHTS = SUBMISSION_ROOT / 'segmentation' / 'weights'
else:
    local_data_candidates = [
        ROOT / 'data' / 'Dataset',
        ROOT / 'Dataset',
        ROOT / 'data',
        ROOT.parent / 'Dataset',
    ]
    DATA_ROOT = next((p for p in local_data_candidates if (p / 'train').exists() and (p / 'test').exists()), ROOT / 'data' / 'Dataset')
    
    local_weights_candidates = [
        ROOT / 'segmentation' / 'weights',
        ROOT / '25942_MAKWANA_JEVINKUMAR_Assignment_2 2' / '25942_MAKWANA_JEVINKUMAR_Assignment_2' / 'segmentation' / 'weights',
        ROOT.parent / 'segmentation' / 'weights',
    ]
    SEG_WEIGHTS = next((p for p in local_weights_candidates if (p / 'best_segmentation_a.pth').exists()), ROOT / 'segmentation' / 'weights')
    SUBMISSION_ROOT = ROOT

SEED = 42
rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 21

print('Using device:', device)
print('Segmentation weights dir:', SEG_WEIGHTS)

## Hypotheses (Wilcoxon Signed-Rank Test)

### Paired Difference Definition
For each validation image $i$, define the paired performance difference as:
$$
d_i = \mathrm{mIoU}_{A,i} - \mathrm{mIoU}_{B,i}
$$
where $\mathrm{mIoU}_{A,i}$ and $\mathrm{mIoU}_{B,i}$ are computed on the same image $i$ (paired design).

### Formal Statistical Hypotheses
Let $m_d$ denote the population median of paired differences $d_i$.

- Null hypothesis $H_0$: $m_d = 0$.
  Interpretation: there is no systematic median performance advantage of Model A over Model B on the target image distribution.
- Alternative hypothesis $H_1$: $m_d \neq 0$ (two-sided).
  Interpretation: a systematic median performance difference exists between Model A and Model B.

At significance level $\alpha = 0.05$, reject $H_0$ if the two-sided Wilcoxon p-value is less than $\alpha$; otherwise, fail to reject $H_0$.

### Why Wilcoxon Over Paired t-Test
1. The paired t-test targets differences in means under approximate normality of $d_i$, while Wilcoxon targets differences in medians/ranks and does not require normality.
2. Per-image IoU differences are frequently skewed or heavy-tailed because many images are easy (near-ceiling IoU) while a smaller subset is difficult.
3. IoU is bounded in $[0, 1]$, which often induces non-Gaussian residual structure and compresses one side of the distribution.
4. Semantic segmentation metrics can contain occasional large failures (outliers) from hard scenes; Wilcoxon is less sensitive to such magnitude outliers because it uses ranks of absolute differences.
5. Sample sizes in validation splits can be moderate, where normal approximations for mean-based tests may be less stable if assumptions are violated.
6. Wilcoxon naturally leverages paired information by testing signed ranks of within-image differences, which is aligned with this A-vs-B same-image comparison protocol.
7. In model-selection settings, robustness to assumption violations is usually preferred over maximum efficiency under perfect Gaussian assumptions; Wilcoxon is the safer default here.

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_paths: List[Path], mask_paths: List[Path], image_size: int = 448):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        mask = Image.open(self.mask_paths[idx])

        img = TF.resize(img, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.NEAREST)

        x = TF.to_tensor(img)
        x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        y = torch.from_numpy(np.asarray(mask, dtype=np.uint8).astype(np.int64))
        return x, y


class ConvBNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: Optional[int] = None):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SegmentationBackbone(nn.Module):
    def __init__(self, pretrained_backbone: bool = False):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained_backbone else None
        backbone = models.resnet50(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


class SegmentationDecoder(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1, padding=0)
        self.fuse34 = nn.Sequential(
            ConvBNAct(256 + 1024, 256),
            ConvBNAct(256, 256),
        )
        self.fuse23 = nn.Sequential(
            ConvBNAct(256 + 512, 192),
            ConvBNAct(192, 192),
        )
        self.fuse12 = nn.Sequential(
            ConvBNAct(192 + 256, 128),
            ConvBNAct(128, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, c1: torch.Tensor, c2: torch.Tensor, c3: torch.Tensor, c4: torch.Tensor, out_size: Tuple[int, int]) -> torch.Tensor:
        x = self.proj4(c4)
        x = F.interpolate(x, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse34(torch.cat([x, c3], dim=1))
        x = F.interpolate(x, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse23(torch.cat([x, c2], dim=1))
        x = F.interpolate(x, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse12(torch.cat([x, c1], dim=1))
        x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained_backbone: bool = False):
        super().__init__()
        self.backbone = SegmentationBackbone(pretrained_backbone=pretrained_backbone)
        self.decoder = SegmentationDecoder(num_classes=num_classes)

    def forward(self, x: torch.Tensor):
        c1, c2, c3, c4 = self.backbone(x)
        logits = self.decoder(c1, c2, c3, c4, x.shape[-2:])
        return {'out': logits}


def split_seg_data(val_ratio: float = 0.05, seed: int = 42):
    img_dir = DATA_ROOT / 'train' / 'images'
    mask_dir = DATA_ROOT / 'train' / 'segmentation_masks'
    img_paths = sorted(img_dir.glob('*.jpg'))
    pairs = []
    for p in img_paths:
        mp = mask_dir / f'{p.stem}.png'
        if mp.exists():
            pairs.append((p, mp))

    if not pairs:
        raise FileNotFoundError(f'No image/mask pairs found under {img_dir} and {mask_dir}.')

    rs = np.random.RandomState(seed)
    rs.shuffle(pairs)
    n_val = max(1, int(len(pairs) * val_ratio))
    va = pairs[:n_val]

    va_i = [a for a, _ in va]
    va_m = [b for _, b in va]
    return va_i, va_m


def miou_per_image(pred: np.ndarray, gt: np.ndarray) -> float:
    valid = gt != 255
    ious = []
    for c in range(1, NUM_CLASSES):
        p = (pred == c) & valid
        g = (gt == c) & valid
        u = np.logical_or(p, g).sum()
        if u > 0:
            i = np.logical_and(p, g).sum()
            ious.append(float(i) / float(u))
    return float(np.mean(ious)) if ious else 0.0


def load_seg_model(ckpt_path: Path) -> nn.Module:
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')
    model = CustomSegmentationNet(num_classes=NUM_CLASSES, pretrained_backbone=False).to(device)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if isinstance(ckpt, dict):
        state = ckpt.get('model_state', ckpt.get('state_dict', ckpt))
    else:
        state = ckpt
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

In [ ]:
ckpt_a = SEG_WEIGHTS / 'best_segmentation_a.pth'
ckpt_b = SEG_WEIGHTS / 'best_segmentation_b.pth'

model_a = load_seg_model(ckpt_a)
model_b = load_seg_model(ckpt_b)

val_ratio = 0.05
val_seed = SEED + 123
eval_image_size = 448

va_i, va_m = split_seg_data(val_ratio=val_ratio, seed=val_seed)
image_ids = [p.stem for p in va_i]
if len(va_i) == 0:
    raise RuntimeError('Validation split is empty. Check dataset paths and val_ratio.')

val_loader = DataLoader(SegmentationDataset(va_i, va_m, image_size=eval_image_size), batch_size=1, shuffle=False, num_workers=0)

scores_a = []
scores_b = []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        gt = y.squeeze(0).numpy().astype(np.uint8)

        logits_a = model_a(x)['out']
        logits_a_flip = torch.flip(model_a(torch.flip(x, dims=[3]))['out'], dims=[3])
        pa = torch.argmax(0.5 * (logits_a + logits_a_flip), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

        logits_b = model_b(x)['out']
        logits_b_flip = torch.flip(model_b(torch.flip(x, dims=[3]))['out'], dims=[3])
        pb = torch.argmax(0.5 * (logits_b + logits_b_flip), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

        scores_a.append(miou_per_image(pa, gt))
        scores_b.append(miou_per_image(pb, gt))

scores_a = np.asarray(scores_a, dtype=np.float64)
scores_b = np.asarray(scores_b, dtype=np.float64)
diffs = scores_a - scores_b

print('Validation setup -> val_ratio:', val_ratio, '| val_seed:', val_seed, '| image_size:', eval_image_size)
print('Model A mean mIoU:', float(np.mean(scores_a)))
print('Model B mean mIoU:', float(np.mean(scores_b)))
print('Mean paired difference (A - B):', float(np.mean(diffs)))

per_image_df = pd.DataFrame({
    'image_id': image_ids,
    'miou_a': scores_a,
    'miou_b': scores_b,
    'diff_a_minus_b': diffs,
})
display(per_image_df.head(10))

In [ ]:
def _average_ranks(values: np.ndarray) -> np.ndarray:
    order = np.argsort(values)
    ranks = np.zeros(len(values), dtype=np.float64)
    i = 0
    while i < len(values):
        j = i
        while j + 1 < len(values) and values[order[j + 1]] == values[order[i]]:
            j += 1
        rank = 0.5 * (i + j) + 1.0
        for k in range(i, j + 1):
            ranks[order[k]] = rank
        i = j + 1
    return ranks

def wilcoxon_signed_rank(d: np.ndarray) -> Tuple[float, float, float]:
    d = np.asarray(d, dtype=np.float64)
    d = d[d != 0.0]
    n = len(d)
    if n == 0:
        return 0.0, 1.0, 0.0

    abs_d = np.abs(d)
    ranks = _average_ranks(abs_d)

    w_plus = float(np.sum(ranks[d > 0]))
    w_minus = float(np.sum(ranks[d < 0]))
    w = min(w_plus, w_minus)

    mean_w = n * (n + 1) / 4.0
    var_w = n * (n + 1) * (2 * n + 1) / 24.0
    if var_w <= 0:
        return w, 1.0, 0.0

    z = (w - mean_w) / math.sqrt(var_w)
    p_two = 2.0 * (0.5 * math.erfc(abs(z) / math.sqrt(2.0)))
    p_two = min(1.0, max(0.0, p_two))
    return w, p_two, z

stat, pval, zscore = wilcoxon_signed_rank(diffs)
print(f'Wilcoxon signed-rank statistic: {stat:.4f}')
print(f'Wilcoxon two-sided p-value: {pval:.6f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1) Paired-difference histogram
axes[0, 0].hist(diffs, bins=25, color='#4C78A8', edgecolor='black', alpha=0.85)
axes[0, 0].axvline(float(np.mean(diffs)), color='crimson', linestyle='--', linewidth=2, label=f'mean={float(np.mean(diffs)):.4f}')
axes[0, 0].axvline(0.0, color='black', linestyle=':', linewidth=1.5)
axes[0, 0].set_title('Histogram of Paired Differences (mIoU_A - mIoU_B)')
axes[0, 0].set_xlabel('Difference')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# 2) Model score distributions
axes[0, 1].boxplot([scores_a, scores_b], patch_artist=True,
                 boxprops=dict(facecolor='#A0CBE8'), medianprops=dict(color='darkred', linewidth=2))
axes[0, 1].set_xticks([1, 2])
axes[0, 1].set_xticklabels(['Model A', 'Model B'])
axes[0, 1].set_title('Per-image mIoU Distribution by Model')
axes[0, 1].set_ylabel('mIoU')
axes[0, 1].grid(alpha=0.25, axis='y')

# 3) Paired scatter
axes[1, 0].scatter(scores_a, scores_b, s=22, alpha=0.75, color='#59A14F', edgecolors='black', linewidths=0.3)
mn = float(min(scores_a.min(), scores_b.min()))
mx = float(max(scores_a.max(), scores_b.max()))
axes[1, 0].plot([mn, mx], [mn, mx], linestyle='--', color='gray', linewidth=1.2)
axes[1, 0].set_title('Paired mIoU Scatter: A vs B')
axes[1, 0].set_xlabel('mIoU A')
axes[1, 0].set_ylabel('mIoU B')
axes[1, 0].grid(alpha=0.25)

# 4) Difference by rank to inspect spread and outliers
sorted_d = np.sort(diffs)
axes[1, 1].plot(np.arange(1, len(sorted_d) + 1), sorted_d, color='#F28E2B', linewidth=2)
axes[1, 1].axhline(0.0, color='black', linestyle=':', linewidth=1.2)
axes[1, 1].set_title('Sorted Paired Differences')
axes[1, 1].set_xlabel('Sample rank')
axes[1, 1].set_ylabel('mIoU_A - mIoU_B')
axes[1, 1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## Bootstrap CI (B=1000) for Best Model mIoU

In [ ]:
best_name = 'A' if float(np.mean(scores_a)) >= float(np.mean(scores_b)) else 'B'
best_scores = scores_a if best_name == 'A' else scores_b

B = 1000
n = len(best_scores)
boot_means = []
for _ in range(B):
    idx = rng.integers(0, n, size=n)
    boot_means.append(float(np.mean(best_scores[idx])))

boot_means = np.asarray(boot_means, dtype=np.float64)
overall_miou = float(np.mean(best_scores))
boot_mean = float(np.mean(boot_means))
boot_se = float(np.std(boot_means, ddof=1))
ci_low = float(np.percentile(boot_means, 2.5))
ci_high = float(np.percentile(boot_means, 97.5))

print(f'Best model based on validation mean mIoU: {best_name}')
print(f'Overall mIoU (best model): {overall_miou:.4f}')
print(f'Bootstrap mean: {boot_mean:.4f}')
print(f'Bootstrap standard error: {boot_se:.4f}')
print(f'95% CI (percentile, B=1000): [{ci_low:.4f}, {ci_high:.4f}]')

plt.figure(figsize=(9, 4.8))
plt.hist(boot_means, bins=35, color='#76B7B2', edgecolor='black', alpha=0.85)
plt.axvline(overall_miou, color='darkred', linestyle='--', linewidth=2, label=f'overall={overall_miou:.4f}')
plt.axvline(ci_low, color='black', linestyle=':', linewidth=1.4, label=f'CI low={ci_low:.4f}')
plt.axvline(ci_high, color='black', linestyle=':', linewidth=1.4, label=f'CI high={ci_high:.4f}')
plt.title('Bootstrap Distribution of Mean mIoU (Best Model)')
plt.xlabel('Bootstrap sample mean mIoU')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame([
    {
        'wilcoxon_statistic': stat,
        'wilcoxon_pvalue': pval,
        'wilcoxon_zscore': zscore,
        'modelA_mean_miou': float(np.mean(scores_a)),
        'modelB_mean_miou': float(np.mean(scores_b)),
        'diff_mean_A_minus_B': float(np.mean(diffs)),
        'best_model': best_name,
        'bootstrap_B': 1000,
        'bootstrap_mean': boot_mean,
        'bootstrap_se': boot_se,
        'bootstrap_ci_low': ci_low,
        'bootstrap_ci_high': ci_high,
        'val_ratio': val_ratio,
        'val_seed': val_seed,
        'eval_image_size': eval_image_size,
    }
])

out_csv = SUBMISSION_ROOT / 'statistical_test_results.csv'
summary.to_csv(out_csv, index=False)
print('Saved statistical summary:', out_csv)
summary